# Snavely Reprojection Error

A walkthrough of the reprojection-error cost function used by Noah Snavely's
Bundler and the Ceres BAL examples, plus the coordinate-frame conventions you
need to keep consistent when feeding it from an incremental SfM pipeline.

**Outline**

1. The Ceres cost-function code
2. Anatomy of the cost function
3. General projection formula
4. Working in a common world frame
5. Converting points from one camera's frame to the world frame


## 1. The Ceres Cost-Function Code

```cpp
struct SnavelyReprojectionError {
  SnavelyReprojectionError(double observed_x, double observed_y)
      : observed_x(observed_x), observed_y(observed_y) {}

  template <typename T>
  bool operator()(const T* const camera,
                  const T* const point,
                  T* residuals) const {
    // camera[0,1,2] are the angle-axis rotation.
    T p[3];
    AngleAxisRotatePoint(camera, point, p);
    // camera[3,4,5] are the translation.
    p[0] += camera[3];
    p[1] += camera[4];
    p[2] += camera[5];

    // Compute the center of distortion. The sign change comes from the
    // camera model that Noah Snavely's Bundler assumes, whereby the
    // camera coordinate system has a negative z axis.
    const T xp = -p[0] / p[2];
    const T yp = -p[1] / p[2];

    // Apply second and fourth order radial distortion.
    const T& l1 = camera[7];
    const T& l2 = camera[8];
    const T r2 = xp * xp + yp * yp;
    const T distortion = T(1.0) + r2 * (l1 + l2 * r2);

    // Compute final projected point position.
    const T& focal = camera[6];
    const T predicted_x = focal * distortion * xp;
    const T predicted_y = focal * distortion * yp;

    // The error is the difference between the predicted and observed position.
    residuals[0] = predicted_x - T(observed_x);
    residuals[1] = predicted_y - T(observed_y);
    return true;
  }

  // Factory to hide the construction of the CostFunction object from
  // the client code.
  static ceres::CostFunction* Create(const double observed_x,
                                     const double observed_y) {
    return (new ceres::AutoDiffCostFunction<SnavelyReprojectionError, 2, 9, 3>(
        new SnavelyReprojectionError(observed_x, observed_y)));
  }

  double observed_x;
  double observed_y;
};
```


## 2. Anatomy of the Cost Function

`SnavelyReprojectionError` is the standard reprojection-error cost used in
Ceres Solver examples. It models the difference between observed and projected
pixel coordinates.

### 2.1 Observed pixel measurements

`observed_x`, `observed_y` are the measured pixel coordinates in the image,
provided when the struct is constructed.

### 2.2 Camera parameters (9-dimensional)

The `camera` parameter is a 9-element array:

- `[0..2]`: rotation in angle-axis form,
- `[3..5]`: translation vector,
- `[6]`: focal length,
- `[7..8]`: radial distortion coefficients (second-order $l_1$, fourth-order $l_2$).

### 2.3 Point parameters (3-dimensional)

The `point` parameter is a 3-element array holding the 3D world point.

### 2.4 Projection pipeline

1. **Rotation.** Rotate the 3D point using the angle-axis representation in `camera[0..2]`.
2. **Translation.** Translate the rotated point by `camera[3..5]`.
3. **Perspective division.** Convert to normalized image coordinates (with the Bundler sign convention):

$$
x_p = -\frac{p_0}{p_2}, \qquad y_p = -\frac{p_1}{p_2}
$$

4. **Radial distortion.** Compute

$$
r^2 = x_p^2 + y_p^2, \qquad d(r^2) = 1 + l_1\,r^2 + l_2\,r^4
$$

so that the distorted coordinates are

$$
x_p' = d(r^2)\,x_p, \qquad y_p' = d(r^2)\,y_p
$$

5. **Pixel scaling.** Multiply by the focal length:

$$
\hat{x} = f\,x_p', \qquad \hat{y} = f\,y_p'
$$

### 2.5 Residual

The reprojection error is the difference between predicted and observed pixel
coordinates:

$$
r_x = \hat{x} - x_{\text{obs}}, \qquad r_y = \hat{y} - y_{\text{obs}}
$$

### 2.6 The principal point in the Bundler model

`SnavelyReprojectionError` does not include the principal-point offsets $c_x$
and $c_y$ when computing pixel coordinates. The full Bundler camera model is

$$
\text{pixel} = \begin{bmatrix} f & 0 & c_x \\ 0 & f & c_y \\ 0 & 0 & 1 \end{bmatrix} \cdot \text{normalized}
$$

where $f$ is the focal length and $(c_x, c_y)$ is the principal point. In this
implementation the predicted coordinates are

$$
\hat{x} = f \cdot x_p', \qquad \hat{y} = f \cdot y_p'
$$

with no explicit addition of $c_x, c_y$ (which would otherwise appear as $+c_x,
+c_y$ on the right). The implementation therefore assumes the principal point
sits at the image origin, i.e. $c_x = c_y = 0$. This is a simplification and
may not be valid for every camera. The standard practice is to subtract the
principal point from the observation before passing it in:

$$
x_{\text{obs}} \leftarrow u - c_x, \qquad y_{\text{obs}} \leftarrow v - c_y
$$

so that the cost function still operates on principal-point-centered coordinates.


### 2.7 Why angle-axis (Rodrigues) instead of quaternions?

The cost function stores the rotation as a 3-vector $\boldsymbol{\omega} = \theta\,\hat{\mathbf{n}}$
(angle-axis / Rodrigues), and rotates points with `ceres::AngleAxisRotatePoint`.
Several BAL / SfM pipelines have followed the same convention since Snavely's
Bundler. The reasons:

1. **Minimal, unconstrained parameterization.** Angle-axis is 3-D on
   $\mathbb{R}^3$ with no constraint to maintain. Levenberg–Marquardt can
   update it freely. Quaternions are 4-D with the constraint $\|\mathbf{q}\| = 1$,
   which must be re-enforced (renormalization, or a Ceres `Manifold` /
   `LocalParameterization`).

2. **Smaller per-camera block.** With angle-axis, each camera block is 6 DOF
   in `SnavelyReprojectionErrorFixedCamera` (or 9 DOF in the standard
   `SnavelyReprojectionError` including focal and distortion). Quaternions
   would push these to 7 / 10. For BAL problems with $10^4$–$10^6$ cameras
   this inflates both the Jacobian and the Schur complement.

3. **Direct fit with `AutoDiffCostFunction`.** `AngleAxisRotatePoint` is
   templated on the scalar type and works with Ceres Jets out of the box.
   The cost functor is just a single 9-D (or 6-D) flat parameter block —
   no manifold registration is required.

4. **No practical singularity issue.** Angle-axis is singular at $\theta = \pi$
   (antipodal wrap). In a converging BA iteration $\|\Delta\boldsymbol{\omega}\|$
   is far from $\pi$, so this never bites. `AngleAxisRotatePoint` switches
   between Rodrigues for $\theta > \epsilon$ and a Taylor expansion near
   $\theta = 0$, avoiding division by zero at the identity.

5. **Composes with translation as a flat vector.** Storing
   `camera[0..5] = [\boldsymbol{\omega}, \mathbf{t}]` keeps the block
   contiguous in memory and makes the gradient computation trivial. With
   quaternions the layout becomes `[\mathbf{q}, \mathbf{t}]` (size 7) and a
   `ProductManifold(QuaternionManifold, EuclideanManifold(3))` (Ceres ≥ 2.1)
   has to be wired in.

#### When quaternions *would* be preferable

- Large per-step rotation updates without a good warm start.
- Need for SLERP-style interpolation between keyframes.
- Working inside a Lie-group framework that already speaks SO(3) on the
  manifold (e.g. GTSAM's `Pose3` / `Rot3`, `Sophus::SE3`, `manif::SE3`).
- Long-running attitude estimation where renormalization is cheaper than
  switching parameterizations every step.

For visual SfM with small per-iteration updates and millions of residuals,
three Rodrigues components in a flat block is the cheapest, simplest option.
That is the historical Bundler/BAL choice and the reason this cost function
is written the way it is.


## 3. General Projection Formula

Project a 3D point $\mathbf{X} = [X, Y, Z]^T$ into a camera with:

- rotation $R_w^c$ and translation $t_w^c$ representing the pose of the world
  frame expressed in the camera frame,
- focal length $f$,
- radial distortion coefficients $k_1, k_2$.

### 3.1 Transform into the camera frame

$$
\mathbf{X}_c = R_w^c \mathbf{X} + t_w^c
$$

where:

- $\mathbf{X}_c = [X_c, Y_c, Z_c]^T$ is the point in the camera frame,
- $R_w^c$ is the $3\times 3$ rotation matrix,
- $t_w^c = [t_x, t_y, t_z]^T$ is the translation vector.

### 3.2 Normalized image coordinates

$$
x = \frac{X_c}{Z_c}, \qquad y = \frac{Y_c}{Z_c}
$$

### 3.3 Radial distortion

Compute the squared radius and apply the radial-distortion model:

$$
r^2 = x^2 + y^2
$$

$$
x' = x \cdot \bigl(1 + k_1 r^2 + k_2 r^4\bigr), \qquad
y' = y \cdot \bigl(1 + k_1 r^2 + k_2 r^4\bigr)
$$

where $k_1, k_2$ are the first and second radial-distortion coefficients.

### 3.4 Pixel coordinates

Scaling by the focal length (and adding the principal point if it is non-zero):

$$
u = f\,x' + c_x, \qquad v = f\,y' + c_y
$$


## 4. Working in a Common World Frame

In a standard incremental SfM pipeline that uses the Snavely / BAL (Bundle
Adjustment in the Large) reprojection model, **all cameras and all 3D points
must be expressed in a single common world coordinate system** — usually
chosen to be the first camera. Whenever a new camera is added and new points
are triangulated in some local frame, both the camera pose and the new points
must be transformed into the common reference frame before being handed to the
bundle adjuster.

### 4.1 Why a common frame is required

The `SnavelyReprojectionError` cost function applies

```cpp
AngleAxisRotatePoint(camera, point, p);
p[0] += camera[3];
p[1] += camera[4];
p[2] += camera[5];
```

which is

$$
\mathbf{p}' = R\,\mathbf{X} + \mathbf{t}
$$

where:

- $\mathbf{X}$ is the 3D point in **world** coordinates,
- $R, \mathbf{t}$ transform that point from world into the camera frame.

The cost function therefore assumes that `point` lives in the *same* world
coordinate system as every camera's extrinsics $(R, \mathbf{t})$. If newly
triangulated points are stored in the local frame of the camera that
triangulated them, $R\mathbf{X} + \mathbf{t}$ is no longer consistent across
cameras. Everything must be expressed in one frame — typically the first
camera's frame.

### 4.2 Typical incremental SfM workflow

1. **Choose a reference frame.** Pick the first camera as world origin, so
   $R = I$, $\mathbf{t} = \mathbf{0}$.

2. **Add a second camera.**
   - Recover the relative pose with `cv::recoverPose`, giving
     $\mathbf{R}_{1\to 2}$ and $\mathbf{t}_{1\to 2}$ (the transform from camera
     1 into camera 2).
   - In the world frame, camera 2's pose is simply
     $\mathbf{R}_2 = \mathbf{R}_{1\to 2}$, $\mathbf{t}_2 = \mathbf{t}_{1\to 2}$.
   - Triangulate points in the reference frame:

     ```cpp
     triangulatePoints(P1, P2, matched_points1, matched_points2, points4D);
     ```

     With $P_1 = [I \mid \mathbf{0}]$ and $P_2 = [\mathbf{R}_{1\to 2} \mid
     \mathbf{t}_{1\to 2}]$, the resulting 3D points come out in camera 1's
     coordinate system.

3. **Add a third camera.**
   - Match keypoints between camera 2 and camera 3 and run `recoverPose` again
     to obtain $\mathbf{R}_{2\to 3}, \mathbf{t}_{2\to 3}$.
   - To stay in camera 1's world frame, compose the chain:

$$
\mathbf{R}_{1\to 3} = \mathbf{R}_{2\to 3}\,\mathbf{R}_{1\to 2}, \qquad
\mathbf{t}_{1\to 3} = \mathbf{R}_{2\to 3}\,\mathbf{t}_{1\to 2} + \mathbf{t}_{2\to 3}
$$

   - Newly triangulated points returned by `triangulatePoints` in
     this step may live in camera 2's frame; bring them into camera 1's frame:

$$
\mathbf{X}_w = \mathbf{R}_{1\to 2}^{\top}\,\bigl(\mathbf{X}_2 - \mathbf{t}_{1\to 2}\bigr)
$$

     (or whichever inverse formula matches how the projection matrices were
     constructed in OpenCV).

4. **Add cameras and points to the bundle adjuster** in the common world frame.
   Each camera's extrinsics map *world → camera*, and each 3D point is stored
   in the same world frame.

5. **Refine** all parameters via bundle adjustment.

### 4.3 What must be transformed

Maintaining a single global frame means that, for each new camera:

1. The new camera pose is composed into the world frame.
2. Newly triangulated 3D points are transformed into the world frame.
3. Both are then added to the optimization problem.

Existing points do not need to be re-triangulated as long as they are already
expressed in the world frame.

### 4.4 Summary

- Pick camera 1 as world with $\mathbf{R} = I$, $\mathbf{t} = \mathbf{0}$.
- For each new camera $k$:
  1. Compute $\mathbf{R}_k$ and $\mathbf{t}_k$ in the world frame, by composing
     with the pose of a camera already in that frame.
  2. Triangulate new points and convert them to world coordinates.
  3. Add $\mathbf{R}_k, \mathbf{t}_k$ and the world-frame 3D points to the
     bundle adjuster.

Each residual evaluation inside `SnavelyReprojectionError` then computes

$$
\mathbf{p}_{\text{cam}} = \mathbf{R}_k\,\mathbf{X}_w + \mathbf{t}_k
$$

for the world-frame point $\mathbf{X}_w$ and camera $k$'s extrinsics $\{\mathbf{R}_k, \mathbf{t}_k\}$.


## 5. Converting Points from a Camera's Local Frame to the World Frame

Suppose 3D points have been triangulated between camera 2 and camera 3 and
come out expressed in **camera 2's frame**, but the world frame is camera 1's
frame. The conversion is the inverse of the camera-1-to-camera-2 transform: if
`recoverPose` returned $R_{1\to 2}$ and $\mathbf{t}_{1\to 2}$ such that

$$
\mathbf{X}_2 = R_{1\to 2}\,\mathbf{X}_1 + \mathbf{t}_{1\to 2}
$$

then

$$
\mathbf{X}_1 = R_{1\to 2}^{\top}\bigl(\mathbf{X}_2 - \mathbf{t}_{1\to 2}\bigr)
$$

or, equivalently,

$$
R_{2\to 1} = R_{1\to 2}^{\top}, \qquad
\mathbf{t}_{2\to 1} = -R_{1\to 2}^{\top}\,\mathbf{t}_{1\to 2}
$$

### 5.1 What `recoverPose` returns

Calling

```cpp
recoverPose(E, points1, points2, K, R, t);
```

with `points1` from camera 1 and `points2` from camera 2 returns $R$ and $t$
such that a 3D point $\mathbf{X}_1$ in camera 1's coordinates is transformed
to camera 2's coordinates by

$$
\mathbf{X}_2 = R\,\mathbf{X}_1 + t
$$

This is the transform $\,^{2}T_{1} = (R_{1\to 2}, \mathbf{t}_{1\to 2})$.

### 5.2 Camera 1 as world

Choosing camera 1 as the world frame makes camera 2's extrinsic parameters
$(R_2, t_2)$ identical to the relative pose:

$$
\mathbf{X}_2 = R_2\,\mathbf{X}_w + t_2 \qquad (\mathbf{X}_w \equiv \mathbf{X}_1)
$$

No inversion is needed to define camera 2's pose with respect to the world.

### 5.3 Triangulating between camera 2 and camera 3

When camera 3 is added, matching it to camera 2 and calling `recoverPose`
yields $R_{2\to 3}, \mathbf{t}_{2\to 3}$. Calling `triangulatePoints` with
projection matrices that place camera 2 at the origin produces 3D points
expressed in **camera 2's local frame**.

### 5.4 Converting those points to the world frame

Given $\mathbf{X}_2 = R_{1\to 2}\,\mathbf{X}_1 + \mathbf{t}_{1\to 2}$, invert to
get the world-frame coordinates:

$$
\mathbf{X}_1 = R_{1\to 2}^{\top}\bigl(\mathbf{X}_2 - \mathbf{t}_{1\to 2}\bigr)
$$

Or define the camera-2-to-camera-1 transform directly:

$$
R_{2\to 1} = R_{1\to 2}^{\top}, \qquad
\mathbf{t}_{2\to 1} = -R_{1\to 2}^{\top}\,\mathbf{t}_{1\to 2}
$$

so that

$$
\mathbf{X}_1 = R_{2\to 1}\,\mathbf{X}_2 + \mathbf{t}_{2\to 1}
$$

### 5.5 Use in bundle adjustment

Once the new points are in the world (camera 1) frame, they can be added to
the bundle adjuster along with camera 3's world-frame pose. During
optimization:

- each 3D point is stored in world coordinates,
- each camera's extrinsics transform a world point $\mathbf{X}_w$ into that
  camera's local frame:

$$
\mathbf{X}_{\text{cam}} = R_{\text{cam}}\,\mathbf{X}_w + t_{\text{cam}}
$$

Whenever triangulation produces points in the local frame of some intermediate
camera, applying the inverse of the world-to-that-camera transform brings them
back into the world frame and keeps the Snavely cost function consistent.
